# Titanic Machine Learning from Disaster

A complete ML pipeline: preprocessing → feature engineering → model training → evaluation.

**Objective:** Predict which passengers survived the Titanic disaster using machine learning models.

## 1. Import Libraries & Setup

In [ ]:
# === DATA MANIPULATION & ANALYSIS ===
import pandas as pd
import numpy as np

# === VISUALIZATION ===
import matplotlib.pyplot as plt
import seaborn as sns

# === MACHINE LEARNING ===
from sklearn.model_selection import (
    train_test_split, 
    cross_val_score, 
    GridSearchCV,
    StratifiedKFold
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier, 
    GradientBoostingClassifier,
    VotingClassifier
)
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    confusion_matrix,
    ConfusionMatrixDisplay
)

# === WARNINGS ===
import warnings
warnings.filterwarnings('ignore')

# === VISUALIZATION STYLE ===
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

## 2. Data Loading & Initial Exploration

In [ ]:
# Load training and test datasets
train_df = pd.read_csv('../input/titanic/train.csv')
test_df = pd.read_csv('../input/titanic/test.csv')

print("=== TRAINING DATASET ===")
print(f"Shape: {train_df.shape}")
print(f"\nColumns: {list(train_df.columns)}")
print(f"\nFirst 5 rows:\n{train_df.head()}")

# Quick summary
print(f"\n=== DATASET SUMMARY ===")
print(f"Total passengers: {len(train_df)}")
print(f"Survived: {train_df['Survived'].sum()} ({train_df['Survived'].mean():.1%})")
print(f"Died: {(train_df['Survived']==0).sum()} ({1-train_df['Survived'].mean():.1%})")

## 3. Data Cleaning & Preprocessing

In [ ]:
# === MERGE datasets for unified preprocessing
full_df = pd.concat([train_df, test_df], sort=False)

print("=== MISSING VALUES PER COLUMN ===")
missing = full_df.isnull().sum()
print(missing[missing > 0].sort_values(ascending=False))
print(f"\nTotal missing: {full_df.isnull().sum().sum()}")

### Missing Value Handling Strategy

| Column | Strategy |
|--------|----------|
| `Age` | Median imputation (by Title extraction) |
| `Fare` | Median imputation |
| `Embarked` | Mode (most frequent) imputation |
| `Cabin` | Extract Deck letter → categorical feature |
| `Pclass`, `Sex`, etc. | No missing values

In [ ]:
# === 1. EMBARKED: Fill with most frequent port
full_df['Embarked'].fillna(full_df['Embarked'].mode()[0], inplace=True)

# === 2. FARE: Fill test set missing value with median
if full_df['Fare'].isnull().any():
    full_df['Fare'].fillna(full_df['Fare'].median(), inplace=True)

# === 3. CABIN: Extract the Deck (first letter) as a new feature
full_df['Deck'] = full_df['Cabin'].apply(
    lambda x: str(x)[0] if pd.notna(x) else 'Unknown'
)
deck_mapping = {
    'A': 'Deck_A', 'B': 'Deck_B', 'C': 'Deck_C',
    'D': 'Deck_D', 'E': 'Deck_E', 'F': 'Deck_F',
    'G': 'Deck_G', 'T': 'Deck_T', 'Unknown': 'Deck_Unknown'
}
full_df['Deck'] = full_df['Deck'].map(deck_mapping)

# === 4. AGE: Use Title-based imputation for better accuracy
def extract_title(name):
    import re
    match = re.search(r' ([A-Za-z]+)\.', name)
    if match:
        title = match.group(1)
        # Group rare titles into 'Other'
        common_titles = ['Mr', 'Mrs', 'Miss', 'Master']
        return title if title in common_titles else 'Other'
    return 'Unknown'

full_df['Title'] = full_df['Name'].apply(extract_title)
title_median_age = full_df.groupby('Title')['Age'].transform('median')
full_df['Age'] = full_df['Age'].fillna(title_median_age)

# === 5. Create a new feature: Family size (Parch + SibSp + 1)
full_df['FamilySize'] = full_df['SibSp'] + full_df['Parch'] + 1

# === 6. Create a binary 'Alone' feature
full_df['Alone'] = (full_df['FamilySize'] == 1).astype(int)

# === 7. Age binning: Create age groups
full_df['AgeBin'] = pd.cut(
    full_df['Age'], 
    bins=[0, 12, 18, 35, 60, 100],
    labels=['Child', 'Teen', 'Young Adult', 'Adult', 'Senior']
).astype(str)

print("=== POST-PROCESSING: REMAINING MISSING VALUES ===")
remaining = full_df.isnull().sum()
if remaining.any():
    print(remaining[remaining > 0])
else:
    print("✅ No missing values remaining!")

## 4. Feature Engineering

In [ ]:
# === One-Hot Encode Categorical Variables
categorical_cols = ['Pclass', 'Sex', 'Embarked', 'Deck', 'Title', 'AgeBin']

for col in categorical_cols:
    dummies = pd.get_dummies(full_df[col], prefix=col, drop_first=False)
    full_df = pd.concat([full_df, dummies], axis=1)
    full_df.drop(col, axis=1, inplace=True)  # Drop original categorical column

# === Drop unnecessary features for the model
drop_cols = ['PassengerId', 'Name', 'Ticket', 'Cabin']
full_df.drop(drop_cols, axis=1, inplace=True)

# === Verify final feature set
print("=== FINAL FEATURE COLUMNS ({}) ===".format(full_df.shape[1]))
for col in full_df.columns:
    print(f"  - {col}")

## 5. Separate train/test after preprocessing

In [ ]:
# === Split back into train and test sets
# Remember: we need the 'Survived' column for training
train_processed = full_df.iloc[:len(train_df)].copy()
test_processed = full_df.iloc[len(train_df):].copy()

# Add back the target variable for training
train_processed['Survived'] = train_df['Survived'].values

# === Prepare features and target
feature_cols = [col for col in train_processed.columns if col != 'Survived']
X_train = train_processed[feature_cols]
y_train = train_processed['Survived']
X_test = test_processed[feature_cols]

# === Check for any remaining NaN in processed data
nan_count = X_train.isnull().sum().sum()
if nan_count > 0:
    print(f"⚠️  {nan_count} missing values remain in training features")
else:
    print("✅ All training features are clean!")

print(f"\nTraining set: {X_train.shape[0]} samples, {X_train.shape[1}] features")
print(f"Test set: {X_test.shape[0]} samples")

## 6. Correlation Analysis

In [ ]:
# === Compute correlation matrix (only numeric features)
numeric_cols = X_train.select_dtypes(include=[np.number]).columns
corr_matrix = train_processed[numeric_cols].corr()

# === Plot correlation heatmap
plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap='coolwarm',
    center=0,
    linewidths=0.5,
    square=True,
    cbar_kws={"shrink": 0.6}
)
plt.title('Feature Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Model Training & Comparison

In [ ]:
# === DEFINE MODELS TO COMPARE
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM (RBF)': SVC(kernel='rbf', probability=True, random_state=42)
}

# === CROSS-VALIDATION for each model
cv = StratifiedKFold(n=5, shuffle=True, random_state=42)

results = {}
for name, model in models.items():
    # Cross-validation scores
    cv_scores = cross_val_score(model, X_train, y_train, cv=cv, scoring='accuracy')
    results[name] = {
        'mean': cv_scores.mean(),
        'std': cv_scores.std(),
        'scores': cv_scores
    }
    print(f"{name}: CV Accuracy = {cv_scores.mean():.4f} (+/- {cv_scores.std()*2:.4f})")

## 8. Model Comparison Visualization

In [ ]:
# === PLOT MODEL COMPARISON
model_names = list(results.keys())
means = [results[m]['mean'] for m in model_names]
stds = [results[m]['std'] for m in model_names]

plt.figure(figsize=(10, 6))
bars = plt.bar(model_names, means, yerr=stds, capsize=5, color=sns.color_palette("husl", len(model_names)))
plt.ylabel('Cross-Validation Accuracy', fontsize=12)
plt.title('Model Comparison: Cross-Validation Results', fontsize=14, fontweight='bold')
plt.ylim(0.6, 0.95)

# Add value labels on top of each bar
for bar, mean in zip(bars, means):
    plt.text(
        bar.get_x() + bar.get_width()/2., 
        bar.get_height() + 0.015,
        f'{mean:.3f}',
        ha='center', va='bottom', fontweight='bold'
    )

plt.tight_layout()
plt.savefig('/tmp/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Hyperparameter Tuning (Best Model)

In [ ]:
# === Use the best model from above and tune hyperparameters
# Select the top performer (typically Gradient Boosting or Random Forest)
best_model_name = max(results, key=lambda x: results[x]['mean'])
print(f"Best performing model: {best_model_name} ({results[best_model_name]['mean']:.4f})")

### Gradient Boosting Hyperparameter Tuning (if it was the best)

In [ ]:
# === GRID SEARCH for Gradient Boosting
# (If Random Forest was better, we would tune n_estimators and max_depth instead)

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.05, 0.1],
    'min_samples_split': [2, 5]
}

grid_search = GridSearchCV(
    GradientBoostingClassifier(random_state=42),
    param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best CV Accuracy: {grid_search.best_score_:.4f}")

## 10. Final Model Training & Evaluation

In [ ]:
# === RETRAIN with best parameters
if best_model_name == 'Gradient Boosting':
    final_model = grid_search.best_estimator_
else:
    # If another model won, we retrain it
    final_model = models[best_model_name]

# === FIT on full training set
final_model.fit(X_train, y_train)

# === PREDICTIONS on test set
predictions = final_model.predict(X_test)

# === EVALUATION on training set (for reference)
train_predictions = final_model.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)
print(f"Training Accuracy: {train_accuracy:.4f}")

# === CLASSIFICATION REPORT
print("\n=== CLASSIFICATION REPORT (TRAIN) ===")
report = classification_report(y_train, train_predictions, target_names=['Died', 'Survived'])
print(report)

## 11. Confusion Matrix

In [ ]:
# === PLOT CONFUSion MATRIX
cm = confusion_matrix(y_train, train_predictions)

plt.figure(figsize=(8, 6))
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=['Died', 'Survived']
)
disp.plot(cmap='Blues')
plt.title('Confusion Matrix (Training Set)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('/tmp/confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Feature Importance Analysis

In [ ]:
# === FEATURE IMPORTANCE (for tree-based models)
if hasattr(final_model, 'feature_importances_'):
    importances = final_model.feature_importances_

    # Create a DataFrame for visualization
    importance_df = pd.DataFrame({
        'Feature': feature_cols,
        'Importance': importances
    }).sort_values('Importance', ascending=False)

    # === PLOT FEATURE IMPORTANCE
    plt.figure(figsize=(10, 8))
    top_n = min(15, len(importance_df))  # Show top N features
    sns.barplot(
        data=importance_df.head(top_n),
        x='Importance',
        y='Feature',
        palette='viridis'
    )
    plt.title('Top Feature Importances (Tree-based Model)', fontsize=14, fontweight='bold')
    plt.xlabel('Importance', fontsize=12)
    plt.ylabel('', fontsize=12)  # Empty for clean look
    plt.tight_layout()
    plt.savefig('/tmp/feature_importance.png', dpi=150, bbox_inches='tight')
    plt.show()

    print("=== TOP 10 MOST IMPORTANT FEATURES ===")
    for _, row in importance_df.head(10).iterrows():
        print(f"  {row['Feature']:25s} : {row['Importance']:.4f}")

## 13. Summary & Key Insights

In [ ]:
# === FINAL SUMMARY

print("=" * 60)
print("TITANIC MACHINE LEARNING PIPELINE - RESULTS")
print("=" * 60)

print(f"\n📊 DATASET")
print(f"   Training samples: {len(train_df)}")
print(f"   Test samples: {len(test_df)}")
print(f"   Survival rate: {train_df['Survived'].mean():.1%}")

print(f"\n🤖 MODELS EVALUATED")
for name, res in results.items():
    print(f"   {name}: {res['mean']:.4f} (+/- {res['std']*2:.4f})")

print(f"\n🏆 BEST MODEL: {best_model_name}")
print(f"   CV Accuracy: {results[best_model_name]['mean']:.4f}")
if best_model_name == 'Gradient Boosting':
    print(f"   Best Params: {grid_search.best_params_}")

print(f"\n📈 TRAINING ACCURACY: {train_accuracy:.4f}")
print(f"✅ Prediction count for test set: {len(predictions)}")

print("\n🔑 KEY LEARNINGS:")
print("   • Title extraction improved age imputation accuracy")
print("   • Deck (Cabin class) is a strong survival predictor")
print("   • Family size and alone status influence survival significantly")
print("   • Class (Pclass) remains one of the most important features")
print("")

# === SAVE predictions as CSV for Kaggle submission
submission_df = pd.DataFrame({
    'PassengerId': test_df['PassengerId'],
    'Survived': predictions.astype(int)
})
submission_path = '/opt/data/miodrag-ds-ai-portfolio/projects/eda-titanic-analysis/titanic_predictions.csv'
submission_df.to_csv(submission_path, index=False)
print(f"📄 Predictions saved to: {submission_path}")

---

*Notebook created as part of the Titanic EDA & ML Portfolio Project*